In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define pesticide concentration based on decision variables
def compute_C(t, x, C0, k, T):
    return sum(x[i] * C0 * np.exp(-k * (t - i * T)) for i in range(len(x)) if t >= i * T)

# Define the agricultural ecosystem ODE
def agricultural_ecosystem_step(P, H, B, t, params, x):
    C_t = compute_C(t, x, params['C0'], params['k'], params['T'])

    dPdt = (params['r1'] / (1 + (params['C_w1'] * C_t)**2)) * (
        P * (1 - P / params['K_P']) - params['w1'] * H * P - params['G'] * P
    )
    dHdt = (params['r2'] * H * (1 - H / (params['w2'] * P))) - params['beta'] * B * H - params['C_w2'] * C_t * H
    dBdt = (params['r3'] * B * (1 - B / (params['w3'] * H)))

    return dPdt, dHdt, dBdt

# Numerical integration using fixed time steps
def solve_ecosystem_numerical(y0, t_span, dt, params, x, harvest_times, harvest_rate):
    P, H, B = y0
    t_values = np.arange(t_span[0], t_span[1] + dt, dt)
    results = {'t': [], 'P': [], 'H': [], 'B': []}

    for t in t_values:
        # Record values at this step
        results['t'].append(t)
        results['P'].append(P)
        results['H'].append(H)
        results['B'].append(B)

        # Check if harvest event occurs
        if any(abs(t - ht) < dt / 2 for ht in harvest_times):
            print(f"Harvest event at time {t:.2f}")
            P *= (1 - harvest_rate)  # Apply harvest

        # Compute derivatives
        dPdt, dHdt, dBdt = agricultural_ecosystem_step(P, H, B, t, params, x)

        # Update variables using Euler's method
        P += dPdt * dt
        H += dHdt * dt
        B += dBdt * dt

        # Prevent negative values
        P = max(P, 0)
        H = max(H, 0)
        B = max(B, 0)

    return results

# Parameters
params = {
    'r1': 0.5,       # Growth rate of crops
    'r2': 0.5,       # Growth rate of pests
    'r3': 0.5,       # Growth rate of beneficial insects
    'w1': 0.05,      # Interaction coefficient between pests and crops
    'w2': 1,         # Interaction coefficient between crops and pests
    'w3': 2,         # Interaction coefficient between pests and beneficial insects
    'beta': 0.03,    # Impact of beneficial insects on pests
    'G': 0.01,       # Seasonal harvesting rate
    'C0': 1.0,       # Initial pesticide concentration
    'C_w1': 100,     # Chemical impact on nutrients
    'C_w2': 0.5,     # Chemical impact on pests
    'k': 0.03,       # Decay rate for pesticide
    'T': 13,         # Pesticide application frequency (quarterly in weeks)
    'K_P': 100,      # Carrying capacity for crops
}

# Initial conditions and settings
y0 = [100, 30, 10]  # Initial populations [P, H, B]
t_span = (0, 208)   # 4 years in quarters
dt = 1              # Time step (1 week)
x = [1.5] * 16      # Constant pesticide input
harvest_times = [39 + k * 52 for k in range(4)]  # Harvest at 3rd quarter each year
harvest_rate = 0.85  # 85% of crops are harvested

# Solve the system
results = solve_ecosystem_numerical(y0, t_span, dt, params, x, harvest_times, harvest_rate)

# Plot the results
plt.figure(figsize=(12, 6))
plt.plot(results['t'], results['P'], label='Crops (P)', color='green', linewidth=2)
plt.plot(results['t'], results['H'], label='Pests (H)', color='red', linewidth=2)
plt.plot(results['t'], results['B'], label='Beneficial Insects (B)', color='blue', linewidth=2)
plt.xlabel('Time (weeks)', fontsize=14)
plt.ylabel('Population', fontsize=14)
plt.title('Population Dynamics with Harvest Events', fontsize=16)
plt.legend(fontsize=12)
plt.grid(alpha=0.5)
plt.tight_layout()
plt.show()
